# FV 02 — Point-in-time fundamentals panel (monthly, 2016–2025)

Protocol: [`discussions/Fair_Value/002`](../../discussions/Fair_Value/002_claude_fv02_panel.md). Data build plus checks; still no signal and no returns.

**Inputs**
- FV 01 EDGAR files.
- Alpaca 1Day bars: adjusted (`bars/1Day`) and **raw** (`bars/1Day_raw`). Run `python tools/prices/daily_download.py --source alpaca --adjustment raw` first.

**Steps**
1. Filing snapshots.
2. Month-end prices.
3. Ticker → company mapping over time.
4. Monthly panel with market cap and EV.
5. Exits table.

**Checks**
- A. Apple values against its reported figures.
- B. Universe per year after filters.
- C. DCF item coverage on the priced universe.
- D. Price coverage of companies that filed a Form 25 (exchange delisting).
- E. Mapping sanity: market cap vs public float, ambiguous tickers.
- F. Exits by type and year.

Heavy steps are cached under `<EQUITY_TRADING_DIR>/EquityTrading/fair_value/`. Delete a file there to rebuild it.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1] / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from equity_trading import edgar as E, fv_panel as F

OUT = Path("02"); OUT.mkdir(exist_ok=True)
FV = F.fv_dir()
e = E.edgar_dir()
print("fv_panel", F.FV_PANEL_VERSION, "| edgar", E.EDGAR_VERSION, "|", FV)
companies = pd.read_parquet(e / "companies.parquet", columns=["cik", "name", "sic"])
filings = pd.read_parquet(e / "filings.parquet", columns=["cik", "accn", "form", "filing_date", "accepted"])
filings = filings[filings["filing_date"] <= "2025-12-31"]
status = E.company_status(filings.assign(form=filings["form"].astype(str)))

fv_panel 0.1.0 | edgar 0.1.2 | D:\MarketData\EquityTrading\fair_value


## 1. Filing snapshots (cached; first build roughly 20–60 min with 3 workers)

In [2]:
summary = F.build_snapshots(workers=3)
if len(summary):
    display(summary.tail())
snap = F.load_snapshots()
print(f"{len(snap):,} snapshots, {snap['cik'].nunique():,} CIKs, accepted {snap['accepted'].min():%Y-%m-%d} .. {snap['accepted'].max():%Y-%m-%d}")

42 of 42 facts parts to process, 3 workers


[Parallel(n_jobs=3)]: Using backend LokyBackend with 3 concurrent workers.
[Parallel(n_jobs=3)]: Done  12 tasks      | elapsed:  4.5min
[Parallel(n_jobs=3)]: Done  42 out of  42 | elapsed:  8.7min finished


,part,ciks,snapshots,seconds
37,part-00037.parquet,154,1136,3.8
38,part-00038.parquet,199,856,2.6
39,part-00039.parquet,175,324,1.5
40,part-00040.parquet,0,0,0.4
41,part-00041.parquet,0,0,0.3


696,581 snapshots, 14,755 CIKs, accepted 1993-08-13 .. 2025-12-31


### A. Apple check (CIK 320193) against reported figures

In [3]:
aapl = snap[snap["cik"] == 320193].set_index("period_end").sort_index()
cols = ["form", "accepted", "ttm_revenue", "ttm_net_income", "ttm_cfo", "ttm_capex", "fy0_revenue", "fy1_revenue",
        "cash", "debt", "shares"]
display(aapl.loc["2022-09-01":"2024-01-31", cols].assign(**{c: (lambda x, c=c: x[c] / 1e9) for c in cols[2:]}))
# Reported (USD bn): FY2023 (Sep-23) revenue 383.285, FY2022 394.328; TTM to Dec-23 = 383.285 + 119.575 - 117.154 = 385.706
checks = [("FY2023 revenue", "2023-09-30", "ttm_revenue", 383.285),
          ("FY2022 revenue (fy1 at the FY2023 10-K)", "2023-09-30", "fy1_revenue", 394.328),
          ("TTM revenue to Dec-2023", "2023-12-30", "ttm_revenue", 385.706)]
for name, pe, col, exp in checks:
    rows = aapl.loc[aapl.index == pe, col]
    got = rows.iloc[-1] / 1e9 if len(rows) else np.nan
    print(f"{name:42s} got {got:9.3f}  expected {exp:9.3f}  {'OK' if abs(got - exp) < 0.01 else 'MISMATCH'}")

,form,accepted,ttm_revenue,ttm_net_income,ttm_cfo,ttm_capex,fy0_revenue,fy1_revenue,cash,debt,shares
period_end,,,,,,,,,,,
2022-09-24,10-K,2022-10-27 22:01:14,394.328,99.803,122.151,10.708,394.328,365.817,23.646,120.069,15.908118
2022-12-31,10-Q,2023-02-02 23:01:30,387.537,95.171,109.190,11.692,394.328,365.817,20.535,111.110,15.821946
2023-04-01,10-Q,2023-05-04 22:03:52,385.095,94.321,109.584,12.094,394.328,365.817,24.687,109.615,15.728702
2023-07-01,10-Q,2023-08-03 22:04:43,383.933,94.760,113.072,12.085,394.328,365.817,28.408,109.280,15.634232
2023-09-30,10-K,2023-11-02 22:08:27,383.285,96.995,110.543,10.959,383.285,394.328,29.965,111.088,15.552752
2023-12-30,10-Q,2024-02-01 23:03:38,385.706,100.913,116.433,9.564,383.285,394.328,40.760,108.040,15.441881


FY2023 revenue                             got   383.285  expected   383.285  OK
FY2022 revenue (fy1 at the FY2023 10-K)    got   394.328  expected   394.328  OK
TTM revenue to Dec-2023                    got   385.706  expected   385.706  OK


## 2. Month-end prices from Alpaca (raw + adjusted)

In [4]:
mp_path = FV / "month_end_prices.parquet"
syms = pd.read_csv(e / "price_symbols.csv")["alpaca"].dropna().unique().tolist()
if mp_path.exists():
    mp = pd.read_parquet(mp_path)
else:
    mp = F.month_end_prices(syms, workers=4)
print(f"{mp['symbol'].nunique():,} symbols with raw bars, {len(mp):,} symbol-months")
if mp["close_adj"].isna().mean() > 0.05:
    print(f"warning: {mp['close_adj'].isna().mean():.1%} of months have no adjusted close (adjusted cache incomplete?)")

[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  42 tasks      | elapsed:    8.0s
[Parallel(n_jobs=4)]: Done 192 tasks      | elapsed:   22.3s
[Parallel(n_jobs=4)]: Done 442 tasks      | elapsed:   39.5s
[Parallel(n_jobs=4)]: Done 880 tasks      | elapsed:  1.2min
[Parallel(n_jobs=4)]: Done 1446 tasks      | elapsed:  2.0min
[Parallel(n_jobs=4)]: Done 1996 tasks      | elapsed:  2.6min
[Parallel(n_jobs=4)]: Done 2646 tasks      | elapsed:  3.4min
[Parallel(n_jobs=4)]: Done 3396 tasks      | elapsed:  4.3min
[Parallel(n_jobs=4)]: Done 4246 tasks      | elapsed:  5.3min
[Parallel(n_jobs=4)]: Done 5196 tasks      | elapsed:  6.5min
[Parallel(n_jobs=4)]: Done 6246 tasks      | elapsed:  7.7min
[Parallel(n_jobs=4)]: Done 7664 tasks      | elapsed:  9.4min
[Parallel(n_jobs=4)]: Done 9122 tasks      | elapsed: 11.1min
[Parallel(n_jobs=4)]: Done 11256 tasks      | elapsed: 13.6min
[Parallel(n_jobs=4)]: Done 12706 tasks      | elapsed: 15.3m

10,818 symbols with raw bars, 741,938 symbol-months


## 3. Ticker → company mapping and 4. monthly panel

In [5]:
th = pd.read_parquet(e / "ticker_history.parquet")
win = F.symbol_windows(th, status)
assigned = F.assign_symbols(win, mp)
print(f"assigned {len(assigned):,} CIK-months, {assigned['cik'].nunique():,} CIKs; "
      f"ambiguous symbol-months dropped: {assigned.attrs['ambiguous_dropped']:,}")
months = pd.date_range("2016-01-31", "2025-12-31", freq="ME")
panel = F.monthly_panel(snap, assigned, companies, months=months)
panel.to_parquet(FV / "panel_monthly.parquet", index=False)
print(f"panel: {len(panel):,} rows, {panel['cik'].nunique():,} CIKs")

assigned 533,004 CIK-months, 8,670 CIKs; ambiguous symbol-months dropped: 14,444
panel: 449,110 rows, 7,203 CIKs


### B. Universe per year (December of each year)

In [6]:
dec = panel[panel["month_end"].dt.month == 12].copy()
dec["year"] = dec["month_end"].dt.year
f_ok = (~dec["is_financial"]) & (~dec["is_blank_check"]) & (dec["mcap"] > 0)
f_map = f_ok & ~dec["mcap_suspect"]
f_dcf = f_map & dec["ttm_revenue"].notna() & dec["ttm_fcf"].notna() & dec["fy4_fcf"].notna() & (dec["ev"] > 0)
u = pd.DataFrame({"priced": dec.groupby("year")["cik"].nunique(),
                  "non_fin_non_spac": dec[f_ok].groupby("year")["cik"].nunique(),
                  "mapping_ok": dec[f_map].groupby("year")["cik"].nunique(),
                  "dcf_ready_5y": dec[f_dcf].groupby("year")["cik"].nunique()})
bins = [-np.inf, 3e8, 2e9, 1e10, 2e11, np.inf]
labels = ["<300M", "300M-2B", "2B-10B", "10B-200B", ">200B"]
evb = dec[f_dcf].assign(ev_bucket=pd.cut(dec.loc[f_dcf, "ev"], bins, labels=labels))
u = u.join(evb.groupby(["year", "ev_bucket"], observed=False)["cik"].nunique().unstack())
u.to_csv(OUT / "universe_by_year.csv"); display(u)

,priced,non_fin_non_spac,mapping_ok,dcf_ready_5y,<300M,300M-2B,2B-10B,10B-200B,>200B
year,,,,,,,,,
2016,3917,2774,2110,1618,408,502,433,262,13
2017,3877,2744,2326,1798,477,535,450,321,15
2018,3922,2769,1966,1490,395,434,381,261,19
2019,3877,2708,1972,1584,349,438,440,338,19
2020,3842,2624,2197,1683,445,462,403,348,25
2021,4141,2750,2342,1679,390,401,450,399,39
2022,3513,2458,1280,934,104,226,310,274,20
2023,3390,2410,1767,1289,172,302,404,376,35
2024,3363,2368,1700,1263,186,303,369,369,36


### C. DCF item coverage on the priced, non-financial universe

In [7]:
items = ["ttm_revenue", "ttm_operating_income", "ttm_net_income", "ttm_cfo", "ttm_capex", "ttm_d_and_a",
         "ttm_interest_expense", "ttm_income_tax", "cash", "debt", "equity", "shares", "fy4_revenue", "fy4_fcf"]
d = dec[f_ok]
cov = d.groupby("year")[items].agg(lambda x: x.notna().mean())
cov["debt_reported"] = d.groupby("year")["debt_reported"].mean()
cov["shares_from_dei"] = d.groupby("year")["shares_source"].agg(lambda x: (x == "dei").mean())
cov.round(3).to_csv(OUT / "item_coverage_priced.csv"); display(cov.round(2))

,ttm_revenue,ttm_operating_income,ttm_net_income,ttm_cfo,ttm_capex,ttm_d_and_a,ttm_interest_expense,ttm_income_tax,cash,debt,equity,shares,fy4_revenue,fy4_fcf,debt_reported,shares_from_dei
year,,,,,,,,,,,,,,,,
2016,0.85,0.87,0.95,0.95,0.88,0.91,0.73,0.81,0.99,1.0,0.97,1.0,0.75,0.83,0.64708,1.00
2017,0.84,0.87,0.95,0.95,0.86,0.91,0.72,0.80,0.99,1.0,0.97,1.0,0.77,0.85,0.651968,0.99
2018,0.83,0.86,0.94,0.93,0.85,0.89,0.70,0.78,0.99,1.0,0.98,1.0,0.77,0.85,0.647165,1.00
2019,0.87,0.88,0.95,0.95,0.87,0.90,0.71,0.78,0.99,1.0,0.98,1.0,0.77,0.85,0.634786,1.00
2020,0.85,0.87,0.94,0.94,0.84,0.89,0.69,0.76,0.99,1.0,0.98,1.0,0.75,0.82,0.63872,0.99
2021,0.79,0.82,0.89,0.88,0.77,0.82,0.65,0.70,0.99,1.0,0.98,1.0,0.71,0.75,0.621091,0.99
2022,0.84,0.88,0.95,0.95,0.84,0.88,0.69,0.78,0.99,1.0,0.98,1.0,0.69,0.73,0.644426,0.99
2023,0.84,0.89,0.96,0.96,0.84,0.89,0.70,0.79,0.99,1.0,0.98,1.0,0.71,0.75,0.635685,0.99
2024,0.84,0.88,0.96,0.96,0.81,0.89,0.71,0.79,0.99,1.0,0.99,1.0,0.74,0.79,0.633024,0.99


### D. Price coverage of exchange-delisted companies (Form 25 / 25-NSE filers, 2016–2025)

In [8]:
f25 = filings[filings["form"].astype(str).isin(["25", "25-NSE"]) & (filings["filing_date"] >= "2016-03-01")]
f25 = f25.sort_values("filing_date").drop_duplicates("cik", keep="last")[["cik", "filing_date"]]
last_px = assigned.groupby("cik")["last_date"].max()
f25["last_price"] = f25["cik"].map(last_px)
f25["covered"] = f25["last_price"].notna() & ((f25["filing_date"] - f25["last_price"]).dt.days <= 120)
f25["periodic_filer"] = f25["cik"].isin(status["cik"])
d25 = f25[f25["periodic_filer"]]
t = d25.groupby(d25["filing_date"].dt.year)["covered"].agg(["mean", "count"])
t.round(3).to_csv(OUT / "form25_price_coverage.csv"); display(t.round(3))
print(f"Form 25 filers with periodic reports: {len(d25):,}; priced up to within 120 days of the form: {d25['covered'].mean():.1%}")

,mean,count
filing_date,,
2016,0.836,189
2017,0.843,235
2018,0.856,180
2019,0.923,195
2020,0.877,155
2021,0.759,224
2022,0.725,182
2023,0.533,212
2024,0.545,211


Form 25 filers with periodic reports: 2,040; priced up to within 120 days of the form: 74.2%


### E. Mapping sanity

In [9]:
pf = panel[panel["mcap_to_float"].notna() & ~panel["is_financial"]]
print(f"rows with public float: {len(pf):,}; mcap/float outside [0.8, 50]: {pf['mcap_suspect'].mean():.1%}")
q = pf["mcap_to_float"].quantile([0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])
display(q.round(2).to_frame("mcap / public float"))
worst = (pf[pf["mcap_suspect"]].groupby(["cik", "name", "symbol"])["mcap_to_float"].median()
         .sort_values().reset_index())
display(pd.concat([worst.head(10), worst.tail(10)]))
worst.to_csv(OUT / "mcap_float_suspects.csv", index=False)
print("shares source:", panel["shares_source"].value_counts(normalize=True).round(3).to_dict())

rows with public float: 304,854; mcap/float outside [0.8, 50]: 26.0%


D:\Anaconda\envs\equity-trading\Lib\site-packages\numpy\lib\_function_base_impl.py:4609: RuntimeWarning: invalid value encountered in subtract
  diff_b_a = b - a


,mcap / public float
0.01,0.00
0.05,0.26
0.25,0.83
0.50,1.23
0.75,1.95
0.95,8.75
0.99,NaN


,cik,name,symbol,mcap_to_float
0,1535527,"CrowdStrike Holdings, Inc.",CRWD,0.0
1,1841156,"Paymentus Holdings, Inc.",PAY,0.0
2,1447599,"FITBIT, INC.",FIT,0.0
3,1853044,"Aeries Technology, Inc.",AERT,0.0
4,1699838,"Confluent, Inc.",CFLT,0.0
5,1650164,"Toast, Inc.",TOST,0.0
6,1846576,"FIGS, Inc.",FIGS,0.0
7,1538379,"Ibotta, Inc.",IBTA,0.0
8,1718227,"Construction Partners, Inc.",ROAD,0.0
9,1759509,"Lyft, Inc.",LYFT,0.0


shares source: {'dei': 0.9, 'none': 0.093, 'gaap': 0.007}


### F. Exits (CIKs whose price stops before 2025-12)

In [10]:
ex = F.exits(filings, mp, assigned)
ex.to_parquet(FV / "exits.parquet", index=False)
t = ex.groupby([ex["last_date"].dt.year, "exit_type"]).size().unstack(fill_value=0)
t.to_csv(OUT / "exits_by_year.csv"); display(t)

exit_type,acquired,delisted_other
last_date,,
2016,208,148
2017,197,106
2018,193,120
2019,187,168
2020,140,398
2021,173,441
2022,180,980
2023,136,164
2024,118,141


## Readout (fill in after the run)

- A. Apple check:
- B. DCF-ready universe per year and by EV bucket:
- C. Weakest items on the priced universe:
- D. Form 25 price coverage:
- E. Mapping suspects (share and examples):
- F. Exits, acquired vs other:
- Decisions for FV 03: universe filters (EV range, mega-cap exclusion), which items are required, delisting-return rule.